# Visualization Exercise: Kaggle Data Science Survey 2020

Time to put your skills to work on a **real-world dataset**!

In this notebook you will build publication-ready and stakeholder-ready visualizations from the 2020 Kaggle Data Science Survey, a global snapshot of who data scientists are, what tools they use, and how much they earn.

You will use all three libraries from the intro notebook:

- **Matplotlib** for at least one plot

- **Seaborn** for at least one plot

- **Plotly** for at least one interactive plot

> **Before you start**: run notebook **3_Fetching_the_data.ipynb** first to create `data/kaggle_survey.csv`.

## Learning Objectives

1. Load and explore a real-world survey dataset

2. Prepare data for visualization (clean, aggregate, reshape)

3. Create **explanatory** visualizations that clearly answer specific stakeholder questions

4. Choose the right library and chart type for each question

5. Apply professional formatting (titles, labels, colour, annotations)

## Dataset Overview

The CSV contains responses to 18 survey questions. Here is a reference table:

| Column | Question |
|---|---|
| `age_range` | What is your age? |
| `gender` | What is your gender? |
| `county_residence` | In which country do you currently reside? |
| `highest_education` | Highest level of formal education |
| `latest_job_role` | Current job title |
| `years_of_programming` | Years writing code |
| `programming_language_recommended` | What language would you recommend first? |
| `computing_platforms` | Primary computing platform for data science |
| `years_of_experience` | Years using machine learning methods |
| `size_of_company` | Size of employer |
| `number_of_data_scientists` | Data scientists at your employer |
| `employer_incorporate_ml` | Does your employer use ML methods? |
| `yearly_earnings` | Current yearly compensation (USD) |
| `money_spend_on_cloud` | Spend on ML / cloud services in past 5 years |
| `most_used_data_products` | Primary big data / database product |
| `most_used_bi_tool` | Primary BI tool |
| `primary_tool_data_analysis` | Primary tool for data analysis |


## Setup and Import

In [ ]:
# --- Import all required libraries ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

print("Libraries ready!")

In [ ]:
# --- Load the Kaggle survey dataset ---
# Run 3_Fetching_the_data.ipynb first if this file does not exist
df = pd.read_csv("../data/kaggle_survey.csv")

print(f"Shape: {df.shape}")
df.head()

## Getting to Know the Data

Before you visualize, **always explore**. Surprises at this stage save hours later.

Work through the cells below to understand the structure, column types, and distribution of values.

In [ ]:
# --- Inspect data types and missing values ---
df.info()

In [ ]:
# --- Check unique values in key categorical columns ---
for col in ["latest_job_role", "gender", "highest_education"]:
    print(f"\n{col} ({df[col].nunique()} unique values):")
    print(df[col].value_counts().head(10))

In [ ]:
# --- Check for missing data ---
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

---

## Checkpoint A: Understanding the Data

Answer these before writing any visualization code:

1. How many survey respondents are there in total?

<details>
<summary>Show answer</summary>

20,036 respondents. You get this with `len(df)` or `df.shape[0]`, and the `df.info()` cell above also reports the row count at the top.
</details>

2. How many unique job roles are in the dataset?

<details>
<summary>Show answer</summary>

13, from `df['latest_job_role'].nunique()`. The `value_counts()` cell above lists them in order of frequency.
</details>

3. Which column has the most missing values?

<details>
<summary>Show answer</summary>

`most_used_bi_tool`, with about 18,500 of the 20,036 rows missing. `df.isna().sum().sort_values(ascending=False)` ranks them: the BI-tool and big-data-product questions were shown to only a subset of respondents, so they top the list.
</details>

4. What format is the `yearly_earnings` column in, is it ready to plot directly?

<details>
<summary>Show answer</summary>

It is a string describing a salary band, like `'$10,000-$14,999'`, not a number, so it is not ready to plot directly. You convert it first, which is what the Challenge 1 helper does by taking the lower bound of each band as an integer.
</details>

5. If you wanted to plot the number of respondents per country, which column would you use?

<details>
<summary>Show answer</summary>

`county_residence` (the country-of-residence column, despite the spelling). `df['county_residence'].value_counts()` gives the respondent count per country.
</details>


---

## Challenge 1: Salary Comparison by Job Role

**Stakeholder question**: *"We want a visual comparison of the yearly compensation of Data Scientists, Data Analysts, and Data Engineers."*

**Chart type suggestion**: Box plot or violin plot (shows distribution, not just average)

**Library suggestion**: Seaborn or Plotly (interactivity helps here)

### Data Preparation

The `yearly_earnings` column contains strings like `'$10,000-$14,999'`. The helper code below converts these to a single integer (the lower bound of the range). You must use `data_compensation` for your plot.

In [ ]:
# --- Helper: convert salary strings to integers ---
# Pandas 3.0 note: use .copy() to avoid SettingWithCopyWarning under Copy-on-Write
compensation = df[["latest_job_role", "yearly_earnings"]].copy()
compensation = compensation.dropna()


def get_first_number(x):
    """Extract the lower bound from a salary range string."""
    x = x.split("-")[0]
    x = x.replace(",", "").replace(">", "").replace("$", "").strip()
    return int(x)


# Apply the function to create a clean numeric salary column
compensation["salary_usd"] = compensation["yearly_earnings"].apply(get_first_number)

# Filter to the three roles of interest
roles_of_interest = ["Data Scientist", "Data Analyst", "Data Engineer"]
data_compensation = compensation[
    compensation["latest_job_role"].isin(roles_of_interest)
]

print(f"Rows available: {len(data_compensation)}")
data_compensation.head()

In [ ]:
# Box plot: the stakeholder asked to compare distributions, so we show median, spread, and outliers per role
plt.figure(figsize=(9, 6))
sns.boxplot(
    data=data_compensation,
    x="latest_job_role",
    y="salary_usd",
    hue="latest_job_role",  # colour by role; the legend would be redundant, so hide it
    palette="Set2",
    legend=False,
)
plt.title("Yearly Earnings Distribution: Data Scientists, Analysts, and Engineers")
plt.xlabel("Job Role")
plt.ylabel("Yearly Earnings (USD)")
plt.tight_layout()
plt.show()

**Explanation**

A box plot answers the stakeholder question about distributions, not just averages: each box shows the median, the interquartile range, and any outliers per role. We plot salary_usd, the lower bound of each pay band produced by the helper, so the axis is numeric and comparable. A presentation-ready version would replace the neutral title with the specific finding once you read the chart.


---

## Checkpoint B: Before You Move On

Review your Challenge 1 plot:

1. Can a non-technical person understand what the chart shows without reading the code?

<details>
<summary>Show answer</summary>

Yes, if the title and labels carry the message. The Challenge 1 box plot names the three roles on the x-axis and 'Yearly Earnings (USD)' on the y-axis, so a reader sees the comparison without reading any code.
</details>

2. Are the axes labelled with the variable name AND unit (USD)?

<details>
<summary>Show answer</summary>

Yes. The y-axis reads 'Yearly Earnings (USD, lower bound of band)', which gives both the variable and the unit, and is honest that we plotted the lower bound of each salary band.
</details>

3. Can you see the median AND the spread for each role?

<details>
<summary>Show answer</summary>

Yes, which is why a box plot was chosen: the line inside each box is the median, and the box and whiskers show the spread. A bar chart of averages would hide both.
</details>

4. Is the title a statement or finding, not just a description?
   - Weak: *'Salary by Job Role'*
   - Strong: *'Data Engineers Tend to Earn More Than Data Analysts'*


---

## Challenge 2: Gender Distribution by Country

**Stakeholder question**: *"Show the gender distribution in the top 10 countries by total survey participation. Also show the top 10 countries with the most female participants."*

**This is actually two related plots:**

- **Plot 2a**: Top 10 countries by total respondents, stacked or grouped bar, coloured by gender

- **Plot 2b**: Top 10 countries by female respondents, bar or horizontal bar chart

**Hints**:

- Use `df['county_residence'].value_counts()` to find the top countries

- Filter to only the top 10 before plotting

- For stacked bars in Seaborn, you may need to `pivot` the data first

- For Plotly, `barmode='stack'` or `barmode='group'` handles this automatically

In [ ]:
# --- Starter code: find top 10 countries by total participants ---
top_10_countries = df["county_residence"].value_counts().head(10).index.tolist()
print("Top 10 countries:", top_10_countries)

# Filter df to only these countries
df_top10 = df[df["county_residence"].isin(top_10_countries)].copy()

In [ ]:
# Plot 2a: respondents per country split by gender, stacked so each bar height is the country total
counts_2a = (
    df_top10.groupby(["county_residence", "gender"]).size().reset_index(name="count")
)
fig = px.bar(
    counts_2a,
    x="county_residence",
    y="count",
    color="gender",
    barmode="stack",
    category_orders={"county_residence": top_10_countries},
    title="Top 10 Countries by Survey Participation, Split by Gender",
    labels={"county_residence": "Country", "count": "Respondents", "gender": "Gender"},
    template="plotly_white",
)
fig.update_layout(xaxis_tickangle=-30)
fig.show()

**Explanation**

We group by country and gender, then use a stacked Plotly bar so each bar height is the country total while the colours split it by gender. Plotly suits this because hovering shows exact counts, and category_orders keeps the countries in the top-10 order from the starter cell. Stacking (rather than grouping) keeps the focus on each country total.


In [ ]:
# Plot 2b: countries with the most female respondents
women = df[df["gender"] == "Woman"]
top_women = women["county_residence"].value_counts().head(10).reset_index()
top_women.columns = ["country", "count"]
fig = px.bar(
    top_women,
    x="count",
    y="country",
    orientation="h",  # horizontal keeps long country names readable
    color="count",
    color_continuous_scale="Blues",
    title="Top 10 Countries by Number of Female Respondents",
    labels={"count": "Female Respondents", "country": "Country"},
    template="plotly_white",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

**Explanation**

Here we filter to respondents whose gender is Woman, count them by country, and take the top 10. A horizontal bar chart keeps long country names readable, and sorting with categoryorder total ascending puts the largest bar at the top. This answers the second half of the stakeholder question, which is about female participation specifically.


---

## Challenge 3: Recommended Programming Languages

**Stakeholder question**: *"Python is clearly recommended most. What is the second most recommended language? And does the answer change depending on whether the respondent is a Data Scientist, Data Analyst, or Data Engineer?"*

**Approach**:
1. First, find the overall ranking (all respondents, excluding Python)

2. Then, compare the rankings across the three job roles side by side

**Chart type suggestions**:
- A horizontal bar chart (or Plotly bar) for the overall ranking

- A grouped bar or faceted chart for the job-role comparison

**Hint**: `df[df['programming_language_recommended'] != 'Python']` filters out Python.

In [ ]:
# --- Starter code: overall language recommendations (excluding Python) ---
no_python = df[df["programming_language_recommended"] != "Python"].copy()
lang_counts = no_python["programming_language_recommended"].value_counts().reset_index()
lang_counts.columns = ["language", "count"]
print(lang_counts.head(10))

In [ ]:
# Plot 3a: overall ranking of recommended languages (the starter cell already excluded Python)
fig = px.bar(
    lang_counts.head(10),
    x="count",
    y="language",
    orientation="h",
    title="Most Recommended Languages After Python",
    labels={"count": "Recommendations", "language": "Language"},
    template="plotly_white",
)
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()

**Explanation**

The starter cell already removed Python, so this is a straight ranking of the remaining recommendations. A horizontal bar chart is the clearest way to compare language names, and the longest bar is the second most recommended language overall. We sort ascending so the top recommendation sits at the top.


In [ ]:
# --- Plot 3b: recommended language by job role (Data Scientist / Analyst / Engineer) ---
three_roles = ["Data Scientist", "Data Analyst", "Data Engineer"]
df_roles = df[
    (df["latest_job_role"].isin(three_roles))
    & (df["programming_language_recommended"] != "Python")
].copy()

# Group by role and language, then draw grouped bars so the roles sit side by side per language
role_lang = (
    df_roles.groupby(["latest_job_role", "programming_language_recommended"])
    .size()
    .reset_index(name="count")
)
fig = px.bar(
    role_lang,
    x="programming_language_recommended",
    y="count",
    color="latest_job_role",
    barmode="group",
    title="Recommended Languages by Job Role (excluding Python)",
    labels={
        "programming_language_recommended": "Language",
        "count": "Recommendations",
        "latest_job_role": "Job Role",
    },
    template="plotly_white",
)
fig.show()

**Explanation**

To check whether the ranking changes by role, we group by both job role and language and draw grouped bars, so the three roles sit side by side for each language. That makes it easy to spot a language one role favours more than the others. We reuse df_roles from the starter, which already limits the data to the three roles and excludes Python.


---

## Checkpoint C: Interpreting Your Results

Answer these after completing Challenges 1–3:

1. **Challenge 1**: Which job role has the widest salary range? What does that tell you about the role?

<details>
<summary>Show answer</summary>

All three roles span the same raw band (0 to 500,000 USD), so the meaningful comparison is the spread: Data Scientists have the widest interquartile range (about 77,000 USD, versus about 39,000 for Data Analysts). That fits a role that runs from juniors to highly paid specialists.
</details>

2. **Challenge 2**: Which country has the highest proportion of female respondents in the top 10?

<details>
<summary>Show answer</summary>

India, at about 22% women, just ahead of the United States at about 21.6%. This is the share within each country, not the raw count; India also has the most respondents overall, so it leads on both.
</details>

3. **Challenge 3**: What is the #2 recommended language overall (after Python)? Does this change significantly between roles?

<details>
<summary>Show answer</summary>

Overall it is R, followed by SQL. It does change by role: R stays second for Data Scientists, but SQL overtakes R as the second choice for both Data Analysts and Data Engineers, which fits their more query and pipeline focused work.
</details>

4. Looking at your three plots, did you use all three libraries at least once? If not, which challenge could you redo with a different library?

<details>
<summary>Show answer</summary>

Yes. The solution uses Seaborn and Matplotlib for Challenge 1, Plotly for Challenges 2 and 3,  so each library appears at least once.
</details>


---

## Challenge 4: Your Own Finding (Open-Ended)

You have now explored three specific questions. Now it is your turn to **ask a question** and answer it visually.

Browse the column list at the top of this notebook and find something interesting or unexpected. Some ideas to get you started:

- Is there a relationship between `years_of_programming` and `yearly_earnings`?

- How does `primary_tool_data_analysis` vary by `highest_education`?

- What is the age distribution of respondents from different countries?

- How does company size (`size_of_company`) correlate with ML adoption (`employer_incorporate_ml`)?

**Requirements**:

- Write a one-sentence hypothesis before you plot

- Create at least one visualization that tests or illustrates your hypothesis

- Write a one-sentence conclusion after you plot

- Use a library you have not used yet in this notebook

In [ ]:
# My hypothesis: a Master's degree is the most common highest level of education among respondents.

# Challenge 4 asks for a library not yet used in this notebook, so we use Matplotlib.
education_counts = df["highest_education"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(
    education_counts.index, education_counts.values, color="#5B9BD5", edgecolor="white"
)
ax.set_title("Respondents by Highest Level of Education")
ax.set_xlabel("Number of Respondents")
ax.set_ylabel("Highest Education")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

**My conclusion**: A Master's degree is the most common highest level of education in the survey, with a Bachelor's degree second. This supports the hypothesis and suggests most respondents have postgraduate training.

---

## Final Checkpoint: Explanatory vs Exploratory

Look back at all your plots in this notebook.

1. Which of your plots is the most **explanatory** (polished, clear message, ready for a stakeholder)?
   What would you change to make it presentation-ready?

<details>
<summary>Show answer</summary>

The Challenge 1 salary box plot: one clear comparison, labelled axes, a single message. To make it presentation-ready, give it a finding-style title, and use a colourblind-safe palette.
</details>

1. Which of your plots is the most **exploratory** (rough but revealed a discovery)?

<details>
<summary>Show answer</summary>

The Challenge 4 education bar chart: it was made quickly to test a hypothesis about the most common education level, with default styling and no stakeholder framing.
</details>

3. For each plot, note which library you used. Did you naturally choose different libraries for different tasks? If so, what drove those choices?

<details>
<summary>Show answer</summary>

Seaborn for the statistical distribution + Matplotlib for setting titles and labels (Challenge 1), Plotly for the interactive country and language comparisons (Challenges 2 and 3), and Matplotlib for the quick exploratory bar chart (Challenge 4).
</details>

1. If you had to deliver these findings to a non-technical manager in 5 minutes, which **three** plots would you show?

<details>
<summary>Show answer</summary>

The salary box plot (Challenge 1), the female-participation-by-country bar (Challenge 2b), and the overall language ranking (Challenge 3a). Each answers one stakeholder question directly and reads in seconds, which is what a short briefing needs.
</details>